# Regression discontinuity: do higher mayoral salaries reduce corruption?

A Bayesian replication, with **CausalPy**, of Klašnja, Fazekas & Alshaibani, *"Revisiting the Link between
Politicians' Salaries and Corruption"* (*British Journal of Political Science*, 2026).

**Setting.** Across 11 EU countries, mayoral salaries jump discretely at population thresholds. Municipalities
just above a threshold pay their mayor more than otherwise-identical municipalities just below it — a textbook
sharp regression-discontinuity design.

- **Outcome** `y` = `cri2`, a procurement corruption-risk index in $[0, 1]$ (composite of six red flags).
- **Running variable** `x` = `margin`, the percent distance in population to the *nearest* salary threshold; the
  cutoff is `0`.
- **Treatment** `treated` = `1` if `x >= 0` (above the threshold ⇒ the mayor receives a higher salary).

**Causal question:** what is the effect on procurement corruption risk of the discrete salary raise a mayor
receives at the threshold — i.e. the discontinuity in `cri2` at `x = 0`?

Data: Harvard Dataverse [doi:10.7910/DVN/TESJMM](https://doi.org/10.7910/DVN/TESJMM). This notebook ships a
compact extract (`data/rd_salaries_corruption.csv.gz`, the unique-threshold sample) so it runs without the
896 MB raw file. See `REPORT.md` for the full written findings and `scripts/` for the end-to-end pipeline.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, arviz as az, pymc as pm, pytensor.tensor as pt
import matplotlib.pyplot as plt
import causalpy as cp

df = pd.read_csv("data/rd_salaries_corruption.csv.gz")  # columns: y, x, citycode, country, year, treated
print(df.shape)
df.head()

## 1. The discontinuity, by eye

Binned means of corruption risk against the running variable. The vertical line is the cutoff; the visible drop
as we cross from "below threshold" (lower salary) to "above" (higher salary) is the effect we want to quantify.
(The first stage is real: in the raw data mean mayoral salary jumps ~14%, from €3,623 to €4,137, at the cutoff.)

In [ ]:
d = df[df.x.abs() < 0.3].copy()
d["bin"] = pd.cut(d.x, np.linspace(-0.3, 0.3, 25))
g = d.groupby("bin", observed=True).agg(x=("x","mean"), y=("y","mean"), n=("y","size")).dropna()
fig, ax = plt.subplots(figsize=(8,5))
for sub, c, lab in [(g[g.x<0], "#2c7fb8", "below (lower salary)"), (g[g.x>=0], "#d95f0e", "above (higher salary)")]:
    ax.scatter(sub.x, sub.y, s=sub.n/3, color=c, label=lab)
for sub, c in [(d[d.x<0], "#2c7fb8"), (d[d.x>=0], "#d95f0e")]:
    z = np.polyfit(sub.x, sub.y, 1); xs = np.linspace(sub.x.min(), sub.x.max(), 50)
    ax.plot(xs, np.polyval(z, xs), color=c, lw=2)
ax.axvline(0, color="k", ls="--", alpha=.6); ax.legend()
ax.set_xlabel("margin to salary threshold (0 = cutoff)"); ax.set_ylabel("corruption risk (cri2)");

## 2. Bayesian RD with CausalPy

`cp.RegressionDiscontinuity` fits a Bayesian local regression on each side of the cutoff and reports the
posterior of the discontinuity. Two practical notes for this dataset:

1. CausalPy expects a **pre-computed 0/1 `treated` column** — it is not auto-created from the threshold.
2. We restrict to a **bandwidth** around the cutoff (`bandwidth=`) and fit a **local linear** model. This is the
   modern RD default; global high-order polynomials are discouraged (Gelman & Imbens, 2019) — we show why below.

We use `bandwidth=0.1115`, the MSE-optimal bandwidth `rdrobust` selects for this sample.

In [ ]:
sk = dict(draws=1000, tune=1000, chains=4, target_accept=0.9, random_seed=42, progressbar=False)
rd = cp.RegressionDiscontinuity(
    df[["x", "y", "treated"]],
    formula="y ~ 1 + x + treated + x:treated",
    treatment_threshold=0.0,
    running_variable_name="x",
    bandwidth=0.1115,
    model=cp.pymc_models.LinearRegression(sample_kwargs=sk),
)
fig, ax = rd.plot()
print(rd.effect_summary().text)

In [ ]:
post = np.asarray(rd.discontinuity_at_threshold).flatten()
print(f"discontinuity in cri2: mean {post.mean():+.3f}, "
      f"94% HDI [{np.percentile(post,3):+.3f}, {np.percentile(post,97):+.3f}], "
      f"P(tau<0) = {(post<0).mean():.3f}")

## 3. How robust is the magnitude? Bandwidth sweep

RD estimates depend on the bandwidth. We sweep a range and record the posterior discontinuity at each. The sign
is invariant (firmly negative everywhere); the magnitude is not.

In [ ]:
rows = []
for bw in [0.05, 0.075, 0.1115, 0.2]:
    r = cp.RegressionDiscontinuity(df[["x","y","treated"]], formula="y ~ 1 + x + treated + x:treated",
            treatment_threshold=0.0, running_variable_name="x", bandwidth=bw,
            model=cp.pymc_models.LinearRegression(sample_kwargs=sk))
    p = np.asarray(r.discontinuity_at_threshold).flatten()
    rows.append(dict(bandwidth=bw, n=int((df.x.abs()<=bw).sum()), tau=round(p.mean(),3),
                     hdi=f"[{np.percentile(p,3):+.3f}, {np.percentile(p,97):+.3f}]", p_neg=round((p<0).mean(),3)))
pd.DataFrame(rows)

## 4. Going beyond a 3rd-order polynomial

### 4a. Why *not* just crank the polynomial order
A common (bad) habit is to fit a high-order **global** polynomial in the running variable. On a fixed window the
discontinuity estimate then swings wildly with the order chosen — the Gelman & Imbens (2019) critique. We
demonstrate the instability rather than rely on it.

In [ ]:
w = df[df.x.abs() < 0.25].sample(n=6000, random_state=42) if (df.x.abs()<0.25).sum()>6000 else df[df.x.abs()<0.25]
specs = {
    "linear":    "y ~ 1 + x + treated + x:treated",
    "quadratic": "y ~ 1 + x + I(x**2) + treated + x:treated + I(x**2):treated",
    "cubic":     "y ~ 1 + x + I(x**2) + I(x**3) + treated + x:treated + I(x**2):treated + I(x**3):treated",
}
for name, fml in specs.items():
    r = cp.RegressionDiscontinuity(w[["x","y","treated"]], formula=fml, treatment_threshold=0.0,
            running_variable_name="x", model=cp.pymc_models.LinearRegression(sample_kwargs={**sk, "chains":2}))
    p = np.asarray(r.discontinuity_at_threshold).flatten()
    print(f"{name:10s} tau = {p.mean():+.3f}  94% HDI [{np.percentile(p,3):+.3f}, {np.percentile(p,97):+.3f}]")

### 4b. A principled, fully Bayesian RD model

Instead of higher-order polynomials we build a richer *local* model that improves on the paper's linear-Gaussian
local regression on four fronts:

1. **Bounded outcome.** `cri2 ∈ [0,1]`, so we use a **Beta likelihood** with a logit link instead of a Gaussian
   (which can predict outside the support). Exact 0/1 values are squeezed in with the Smithson–Verkuilen
   transform.
2. **Triangular kernel weights** inside a bandwidth `h` — the Calonico–Cattaneo–Titiunik weighting, expressed as
   a weighted likelihood via `pm.Potential`.
3. **Hierarchy / clustering.** City random intercepts (the Bayesian analogue of clustering SEs by city) plus
   country fixed effects (`ZeroSumNormal`).
4. **Separate slopes** either side of the cutoff.

The discontinuity is reported back on the natural `cri2` scale (directly comparable to the paper's −0.078).

In [ ]:
H = 0.1115
d = df[df.x.abs() <= H].copy()
N = len(d)
city_codes, city_idx = np.unique(d.citycode.values, return_inverse=True)
ctry_codes, ctry_idx = np.unique(d.country.astype(str).values, return_inverse=True)
nC = len(ctry_codes)
w_kern = 1.0 - np.abs(d.x.values)/H                 # triangular kernel
y2 = (d.y.values*(N-1) + 0.5)/N                      # Smithson-Verkuilen squeeze into (0,1)
x, tr = d.x.values, d.treated.values.astype(float)

with pm.Model(coords={"city": city_codes, "country": ctry_codes}) as m:
    a   = pm.Normal("a", -1.4, 1.0)                 # logit baseline (~0.20)
    tau = pm.Normal("tau", 0.0, 1.0)                # discontinuity (logit scale)
    bL  = pm.Normal("bL", 0.0, 2.0); bR = pm.Normal("bR", 0.0, 2.0)
    cfe = pm.ZeroSumNormal("cfe", sigma=1.0, dims="country")
    sig_city = pm.HalfNormal("sig_city", 1.0)
    city_re = pm.Deterministic("city_re", pm.Normal("z_city", 0, 1, dims="city")*sig_city, dims="city")
    phi = pm.Gamma("phi", alpha=3.0, beta=0.5)       # Beta precision, prior mean 6
    eta = a + tau*tr + bL*x*(1-tr) + bR*x*tr + cfe[ctry_idx] + city_re[city_idx]
    mu = pm.math.invlogit(eta)
    pm.Potential("lik", (pt.as_tensor_variable(w_kern) * pm.logp(pm.Beta.dist(mu*phi, (1-mu)*phi), y2)).sum())
    idata = pm.sample(draws=1000, tune=1000, target_accept=0.9, random_seed=42, progressbar=False)

print("divergences:", int(idata.sample_stats.diverging.sum()))
az.summary(idata, var_names=["a","tau","bL","bR","phi","sig_city"], round_to=3)

In [ ]:
# translate the logit-scale discontinuity back to the cri2 scale, averaged over the
# country composition near the cutoff
inv = lambda z: 1/(1+np.exp(-z))
a_s = idata.posterior["a"].values.flatten()
tau_s = idata.posterior["tau"].values.flatten()
cfe_s = idata.posterior["cfe"].stack(s=("chain","draw")).transpose("s","country").values
cw = np.bincount(ctry_idx, minlength=nC)/N
tau_cri2 = (inv(a_s[:,None]+tau_s[:,None]+cfe_s) - inv(a_s[:,None]+cfe_s)) @ cw
az.plot_posterior(tau_cri2, ref_val=0, hdi_prob=0.94)
print(f"covariate-adjusted discontinuity in cri2: {tau_cri2.mean():+.4f} "
      f"94% HDI [{np.percentile(tau_cri2,3):+.4f}, {np.percentile(tau_cri2,97):+.4f}] "
      f"P(tau<0) = {(tau_cri2<0).mean():.3f}   (paper: -0.078)")

## 5. Validity checks — reported honestly

These checks reveal genuine fragility, which we flag rather than bury.

- **Placebo cutoffs.** A fake cutoff at `-0.3` (below-side only) gives a clean null; a fake cutoff at `+0.3` does
  *not* — there is a spurious jump.
- **Donut RD.** Dropping points within `|x| < 0.02` of the cutoff **flips the sign** of the estimate.

Both symptoms point to the same cause: the running variable is distance to the *nearest* of ~100 thresholds and
is **heaped** at round population numbers — `rdrobust` itself warns *"mass points detected in the running
variable."* Naive placebo/donut perturbations are therefore unreliable here (cf. Kolesár & Rothe, 2018). The
original paper instead validates via covariate balance and a 2018 Romania difference-in-discontinuities.

In [ ]:
# donut RD at the true cutoff
rd_donut = cp.RegressionDiscontinuity(df[["x","y","treated"]], formula="y ~ 1 + x + treated + x:treated",
        treatment_threshold=0.0, running_variable_name="x", bandwidth=0.1115, donut_hole=0.02,
        model=cp.pymc_models.LinearRegression(sample_kwargs=sk))
p = np.asarray(rd_donut.discontinuity_at_threshold).flatten()
print(f"donut RD tau = {p.mean():+.3f}  94% HDI [{np.percentile(p,3):+.3f}, {np.percentile(p,97):+.3f}]  "
      f"(sign flip vs the main estimate — see discussion above)")

## 6. Findings

- The paper's frequentist headline reproduces **exactly** from the raw data (`rdrobust`, unique-threshold sample:
  τ = −0.078, p ≈ 1e-5). See `scripts/02_benchmark_rdrobust.py`.
- The Bayesian RD agrees on **direction**: the discontinuity in corruption risk is **negative with posterior
  probability ≈ 1.0** at every bandwidth, and in the covariate-adjusted hierarchical Beta model
  (≈ −0.044, 94% HDI roughly [−0.082, −0.005], P(τ<0) ≈ 0.98). **Higher mayoral salary ⇒ lower corruption risk.**
- The **magnitude is specification-dependent** (≈ −0.04 to −0.20 unadjusted across bandwidths; −0.078 in the
  paper's adjusted spec; ≈ −0.044 in our adjusted Bayesian model), global polynomials are unstable, and a donut
  deletion flips the sign — all symptoms of mass points in a pooled-threshold running variable.

**Bottom line:** the qualitative claim is directionally credible and survives a genuinely Bayesian treatment;
the precise causal magnitude should be treated with caution and would need the manipulation/heaping diagnostics
and balance/reform validation the authors run. *This replication is independent and not peer-reviewed.*